In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
!ls "/content/drive/My Drive/"

'Colab Notebooks'   convertio.co   GoogleColab	 Старое


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Dense

model = Sequential()
inputShape = (32, 32, 3)
chanDim = -1
    # CONV => RELU => BN => POOL
model.add(Conv2D(8, (5, 5), padding="same",
  input_shape=inputShape))
model.add(Activation("relu"))
model.add(BatchNormalization(axis=chanDim))
model.add(MaxPooling2D(pool_size=(2, 2)))
  # first set of (CONV => RELU => CONV => RELU) * 2 => POOL
model.add(Conv2D(16, (3, 3), padding="same"))
model.add(Activation("relu"))
model.add(BatchNormalization(axis=chanDim))
model.add(Conv2D(16, (3, 3), padding="same"))
model.add(Activation("relu"))
model.add(BatchNormalization(axis=chanDim))
model.add(MaxPooling2D(pool_size=(2, 2)))
# second set of (CONV => RELU => CONV => RELU) * 2 => POOL
model.add(Conv2D(32, (3, 3), padding="same"))
model.add(Activation("relu"))
model.add(BatchNormalization(axis=chanDim))
model.add(Conv2D(32, (3, 3), padding="same"))
model.add(Activation("relu"))
model.add(BatchNormalization(axis=chanDim))
model.add(MaxPooling2D(pool_size=(2, 2)))
  # first set of FC => RELU layers
model.add(Flatten())
model.add(Dense(128))
model.add(Activation("relu"))
model.add(BatchNormalization())
model.add(Dropout(0.5))
# second set of FC => RELU layers
model.add(Flatten())
model.add(Dense(128))
model.add(Activation("relu"))
model.add(BatchNormalization())
model.add(Dropout(0.5))
# softmax classifier
model.add(Dense(7))
model.add(Activation("softmax"))
# return the constructed network architecture
print ("End")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


End


In [ ]:
import matplotlib
matplotlib.use("Agg")
# import the necessary packages
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from keras.callbacks import EarlyStopping
from keras.callbacks import ModelCheckpoint
from sklearn.metrics import classification_report
from skimage import transform
from skimage import exposure
from skimage import io
import matplotlib.pyplot as plt
import numpy as np
import random
import pickle
from sklearn.model_selection import train_test_split

In [ ]:
with open("/content/drive/My Drive/GoogleColab/TrafficSignDetectionCNN/dataNew.pickle", 'rb') as f:
  data = pickle.load(f)
print("Data loaded")

with open("/content/drive/My Drive/GoogleColab/TrafficSignDetectionCNN/labels.pickle", 'rb') as f:
  labels = pickle.load(f)
print("Labels loaded")


(trainX, testX, trainY, testY) = train_test_split(data, labels,
                                                  test_size=0.15,
                                                  random_state=42)

# classTotals = trainY.sum(axis=0)
# classWeight = classTotals.max() / classTotals
# print(type(classWeight))

classTotals = trainY.sum(axis=0)
print(classTotals)
classWeight = classTotals.max() / classTotals
print(classWeight/1)

TestclassTotals = testY.sum(axis=0)
TestclassWeight = TestclassTotals.max() / TestclassTotals
print(TestclassTotals)
print("TestclassWeight")
print(TestclassWeight//1)
f=TestclassTotals.argmin()



# initialize the number of epochs to train for, base learning rate,
# and batch size
NUM_EPOCHS = 30
INIT_LR = 1e-3
BS = 64
# load the label names
target_names= ["Main road", "Give way", "Stop", "Traffic is prohibited", "entry is forbidden", "Rough road", "RoadWork"]
numLabels = len(target_names)

# construct the image generator for data augmentation
opt = Adam(learning_rate=INIT_LR, decay=INIT_LR / (NUM_EPOCHS * 0.5))

model.compile(loss="categorical_crossentropy", optimizer=opt,
	metrics=["accuracy"])
model.summary()


Data loaded
Labels loaded
[1179 1231  462  352  646  235  867]
[1.04410517 1.         2.66450216 3.49715909 1.90557276 5.23829787
 1.41983852]
[231 209  78  68 104  35 153]
TestclassWeight
[1. 1. 2. 3. 2. 6. 1.]


/usr/local/lib/python3.12/dist-packages/keras/src/optimizers/base_optimizer.py:86: UserWarning: Argument `decay` is no longer supported and will be ignored.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 32, 32, 8)      │           608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 32, 32, 8)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32, 32, 8)      │            32 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 8)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 16, 16, 16)     │         1,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 16, 16, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 16, 16, 16)     │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 16, 16, 16)     │         2,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 16, 16, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 16, 16, 16)     │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 16)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 8, 8, 32)       │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 8, 8, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 8, 8, 32)       │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 8, 8, 32)       │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 8, 8, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 8, 8, 32)       │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_5 (Activation)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │             

 Total params: 102,503 (400.40 KB)

 Trainable params: 101,783 (397.59 KB)

 Non-trainable params: 720 (2.81 KB)

In [ ]:
aug = ImageDataGenerator(
	rotation_range=5,
	zoom_range=0.05,
	width_shift_range=0.05,
	height_shift_range=0.05,
	shear_range=0.10,
	horizontal_flip=False,
	vertical_flip=False,
	fill_mode="nearest")

# train the network
print("[INFO] training network...")
print(classWeight)

class_Weight = {0: 1.04410517,
                1: 1.,
                2: 2.66450216,
                3: 3.49715909,
                4: 1.90557276,
                5: 5.23829787,
                6: 1.41983852}

checkpointer = ModelCheckpoint(filepath='/content/drive/My Drive/GoogleColab/TrafficSignDetectionCNN/Best_Sign_Class.keras', verbose=1, save_best_only=True)

H = model.fit(aug.flow(trainX, trainY, batch_size=BS),
										validation_data=(testX, testY),
										steps_per_epoch=trainX.shape[0] // BS,
										epochs=NUM_EPOCHS,
                        shuffle=True,
                        class_weight=class_Weight,
                        callbacks=[checkpointer])


[INFO] training network...
[1.04410517 1.         2.66450216 3.49715909 1.90557276 5.23829787
 1.41983852]
Epoch 1/30
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step - accuracy: 0.3122 - loss: 3.8783
Epoch 1: val_loss improved from None to 2.14924, saving model to /content/drive/My Drive/GoogleColab/TrafficSignDetectionCNN/Best_Sign_Class.keras

Epoch 1: finished saving model to /content/drive/My Drive/GoogleColab/TrafficSignDetectionCNN/Best_Sign_Class.keras
77/77 ━━━━━━━━━━━━━━━━━━━━ 11s 99ms/step - accuracy: 0.4548 - loss: 2.8377 - val_accuracy: 0.1743 - val_loss: 2.1492
Epoch 2/30
 1/77 ━━━━━━━━━━━━━━━━━━━━ 4s 56ms/step - accuracy: 0.6562 - loss: 1.1683

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()



Epoch 2: val_loss did not improve from 2.14924
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6562 - loss: 1.1683 - val_accuracy: 0.1743 - val_loss: 2.1611
Epoch 3/30
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.7224 - loss: 1.1842
Epoch 3: val_loss did not improve from 2.14924
77/77 ━━━━━━━━━━━━━━━━━━━━ 7s 88ms/step - accuracy: 0.7818 - loss: 0.9830 - val_accuracy: 0.1743 - val_loss: 2.7380
Epoch 4/30
 1/77 ━━━━━━━━━━━━━━━━━━━━ 8s 107ms/step - accuracy: 0.9219 - loss: 0.5153
Epoch 4: val_loss did not improve from 2.14924
77/77 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9219 - loss: 0.5153 - val_accuracy: 0.1743 - val_loss: 2.7411
Epoch 5/30
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 0.8793 - loss: 0.5552
Epoch 5: val_loss did not improve from 2.14924
77/77 ━━━━━━━━━━━━━━━━━━━━ 6s 82ms/step - accuracy: 0.8975 - loss: 0.4760 - val_accuracy: 0.1743 - val_loss: 3.0209
Epoch 6/30
 1/77 ━━━━━━━━━━━━━━━━━━━━ 4s 54ms/step - accuracy: 0.9219 - loss: 0.2919
Epoch 6:

In [ ]:
# evaluate the network
print("[INFO] evaluating network...")
predictions = model.predict(testX, batch_size=BS)

print(classification_report(testY.argmax(axis=1),
	predictions.argmax(axis=1), target_names=target_names))

# save the network to disk
model.save("/content/drive/My Drive/GoogleColab/TrafficSignDetectionCNN/AdrNet.keras")

[INFO] evaluating network...
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step
                       precision    recall  f1-score   support

            Main road       1.00      1.00      1.00       231
             Give way       1.00      1.00      1.00       209
                 Stop       1.00      1.00      1.00        78
Traffic is prohibited       1.00      1.00      1.00        68
   entry is forbidden       1.00      0.99      1.00       104
           Rough road       0.97      1.00      0.99        35
             RoadWork       1.00      1.00      1.00       153

             accuracy                           1.00       878
            macro avg       1.00      1.00      1.00       878
         weighted avg       1.00      1.00      1.00       878

